<a href="https://colab.research.google.com/github/aakriti1318/llm-inference-sandbox/blob/main/Finetuning/Finetuning_Unsloth_Ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unsloth

Its a python library to do accelerated fine tuning, by saving a lot of VRAM. It speeds up the training process

In [ ]:
# dataset - /content/sample_data/people_data.json

In [ ]:
!uv init

error: Project is already initialized in `/content` (`pyproject.toml` file exists)


In [ ]:
!uv add unsloth

Resolved 121 packages in 1ms
Installed 102 packages in 260ms
 + accelerate==1.14.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.14.2
 + attrs==26.1.0
 + bitsandbytes==0.50.2
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + click==8.5.0
 + content==0.1.0 (from file:///content)
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.8.0
 + cuda-toolkit==13.0.2
 + cut-cross-entropy==25.1.1
 + datasets==4.3.0
 + diffusers==0.40.0
 + dill==0.4.0
 + docstring-parser==0.18.0
 + filelock==3.32.4
 + frozenlist==1.8.0
 + fsspec==2025.9.0
 + h11==0.16.0
 + hf-transfer==0.1.9
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.29.0
 + idna==3.19
 + importlib-metadata==9.0.1
 + jinja2==3.1.6
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + mdurl==0.1.2
 + mpmath==1.3.0
 + msgspec==0.21.1
 + multidict==6.7.1
 + multiprocess==0.70.16
 + nest-asyncio==1.6.0
 + networkx==3.6.1
 + numpy==2.5.2
 + nvidia

In [ ]:
!uv venv --clear

Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Phi-3-mini-4k-instruct-bnb-4bit',
    max_seq_length = 500,
    dtype = None,
    load_in_4bit = True
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.22: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
import json
from datasets import Dataset

with open('/content/sample_data/people_data.json', 'r', encoding = 'utf-8') as f:
  data = json.load(f)

ds = Dataset.from_list(data)

def to_text(ex):
  resp = ex["response"]
  if not isinstance(resp, str):
    resp = json.dumps(resp, ensure_ascii=False)

  msg = [
      {'role': "user", "content": ex["prompt"]},
      {'role': "assistant", "content": resp}
  ]
  return {
      "text": tokenizer.apply_chat_template(msg, tokenize = False, add_generation_prompt=False)
  }

dataset = ds.map(to_text, remove_columns=ds.column_names)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_alpha = 16,
    lora_dropout = 0.,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth'
)

Unsloth 2026.8.22 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    # peft_config = model.peft_config,
    dataset_text_field = 'text',
    max_seq_length = 500,
    tokenizer = tokenizer,
    args = SFTConfig(
        per_device_train_batch_size = 2, # how many seq do we want to read at once in the GPU
        gradient_accumulation_steps = 4, # how many iteration to accumulate before we take step with the optimizer
        warmup_steps = 10, # for the first 10 steps we are increarsing the lr from 0 to the desired lr
        max_steps = 60,
        num_train_epochs = 3,
        logging_steps = 1,
        output_dir = 'outputs',
        optim = 'paged_adamw_8bit',
    )
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/300 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,942,208 of 3,836,021,760 (0.39% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.544477
2,2.493215
3,2.423814
4,2.516214
5,2.684592
6,2.420348
7,2.599349
8,2.539845
9,2.534979
10,2.352826


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


TrainOutput(global_step=60, training_loss=1.9658782561620076, metrics={'train_runtime': 120.481, 'train_samples_per_second': 3.984, 'train_steps_per_second': 0.498, 'total_flos': 764764103079936.0, 'train_loss': 1.9658782561620076, 'epoch': 1.5866666666666667})

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {
        'role': 'user',
        'content': 'Mike is a 30 yr old programmer, he loves hiking'
    }
]

# Take messages -> apply a chat template -> feed to the model -> get the response -> decode the response

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')

outputs = model.generate(input_ids=inputs, max_new_tokens = 500, use_cache=True, temperature=0.7, do_sample=True, top_p=0.9)

response = tokenizer.batch_decode(outputs)[0]

print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=500) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|user|> Mike is a 30 yr old programmer, he loves hiking<|end|><|assistant|> {"name": "Mike", "age": "30", "job": "programmer", "gender": "male"}<|end|>


In [15]:
model.save_pretrained_gguf('aa-phi3mini4k-dataextraction', tokenizer, quantization_method='q4_k_m', maximum_memory_usage=0.3)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in aa-phi3mini4k-dataextraction/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in aa-phi3mini4k-dataextraction.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors: reconstructing file:   0%|          |  0.00B / 2.65GB            

model-00002-of-00002.safetensors: downloading bytes:           |  0.00B            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:28<00:00, 14.27s/it]


Unsloth: Merge process complete. Saved to `/content/aa-phi3mini4k-dataextraction`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10687-mix-67dfc8b (app-b10687-mix-67dfc8b-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['aa-phi3mini4k-dataextraction_gguf/phi-3-mini-4k-instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unslot

{'save_directory': 'aa-phi3mini4k-dataextraction',
 'gguf_directory': 'aa-phi3mini4k-dataextraction_gguf',
 'gguf_files': ['aa-phi3mini4k-dataextraction_gguf/phi-3-mini-4k-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'aa-phi3mini4k-dataextraction_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [16]:
!nvidia-smi

Sun Aug 30 15:09:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   70C    P0             33W /   72W |    2836MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Ollama

### Ollama Installation

In [17]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,914 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,179 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted 

In [35]:
!nohup ollama serve > ollama.log 2>&1 &

In [36]:
!cat ollama.log

Error: listen tcp 127.0.0.1:11434: bind: address already in use


### Build the local Ollama model

In [27]:
# Run the ollama create command pointing directly to your generated Modelfile.

!ollama create aa-phi3mini4k-dataextraction -f aa-phi3mini4k-dataextraction_gguf/Modelfile

### Run your newly created model

In [30]:
!ollama run aa-phi3mini4k-dataextraction "While strolling through a botanical garden, Igor, now 20 earns a living as a tour guide. He is known among friends for conducting amateur astronomy observations in quiet solitude."

 {"name": "Igor", "age": "20", "job": "tour guide", "gender": "male", "lang
"language": "Russian"}

